In [9]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

#____________________________ part a ______________________________#
import pandas as pd
from sklearn.preprocessing import StandardScaler

data = pd.read_csv('titanic.csv', sep=',')
data.head(10)
#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', None)
#for attr in ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']:
#    print(data[attr].value_counts())

data.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

data['Age'].fillna(data['Age'].median(), inplace=True)
data['Embarked'].fillna(data['Embarked'].mode()[0], inplace=True)

data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})
data['Embarked'] = data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

data.head(10)

data=data.astype(float)

# selecting the features and target variable
features = data[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
target = data['Survived']

# scaling the numeric features using StandardScaler
scaler = StandardScaler()
features_scaled = pd.DataFrame(scaler.fit_transform(features), columns=features.columns)
#print(data.dtypes)

#____________________________ part b ______________________________#
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features_scaled, target, test_size=0.2, random_state=42, stratify=target)

print('Training features shape:', X_train.shape)
print('Training target shape:', y_train.shape)
print('Test features shape:', X_test.shape)
print('Test target shape:', y_test.shape)

#____________________________ part c ______________________________#
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV

clf = DecisionTreeClassifier(random_state=10)
param_grid = {'max_depth': [2, 3, 4, 5, 6, 7, 8], 'min_samples_split': [2, 3, 4, 5, 6, 7, 8]}
grid_search = GridSearchCV(clf, param_grid=param_grid, cv=10, n_jobs=-1)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_
best_score = grid_search.best_score_
print('Best hyperparameters:', best_params)
print('Best score:', best_score)
clf = DecisionTreeClassifier(max_depth=best_params['max_depth'], min_samples_split=best_params['min_samples_split'], random_state=10)
clf.fit(X_train, y_train)
accuracy = clf.score(X_test, y_test)
print('Accuracy on test data:', accuracy)

#____________________________ part d ______________________________#

clf = DecisionTreeClassifier(max_depth=5, random_state=10)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro', zero_division=1)
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print("Accuracy: {:.3f}".format(accuracy))
print("Precision: {:.3f}".format(precision))
print("Recall: {:.3f}".format(recall))
print("F1 score: {:.3f}".format(f1))

#____________________________ part e ______________________________#
from sklearn.tree import export_graphviz
import graphviz

# Exporting the tree to a DOT file
dot_data = export_graphviz(clf, out_file=None, 
                           feature_names=X_train.columns, 
                           class_names=['No', 'Yes'], 
                           filled=True, rounded=True, 
                           special_characters=True)

# Visualizing the tree using Graphviz
graph = graphviz.Source(dot_data)
graph.render(filename='tree', format='svg')

#____________________________ part f ______________________________#
from sklearn.model_selection import cross_val_score
import numpy as np 

# Building a decision tree with no pruning
tree = DecisionTreeClassifier(random_state=10)
tree.fit(X_train, y_train)

# Finding the optimal value of ccp_alpha using cross-validation
path = tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
scores = []
for alpha in ccp_alphas:
    pruned_tree = DecisionTreeClassifier(random_state=10, ccp_alpha=alpha)
    _=pruned_tree.fit(X_train, y_train)
    score = cross_val_score(pruned_tree, X_train, y_train, cv=10).mean()
    scores.append(score)

optimal_alpha = ccp_alphas[np.argmax(scores)]

# Prune the tree using the optimal value of ccp_alpha
pruned_tree = DecisionTreeClassifier(random_state=10, ccp_alpha=optimal_alpha)
pruned_tree.fit(X_train, y_train)

pruned_tree_acc = pruned_tree.score(X_test, y_test)
print('Accuracy of pruned tree on the test data: {}'.format(pruned_tree_acc))

#____________________________ part g ______________________________#

# Exporting the pruned tree to a DOT file
dot_data = export_graphviz(pruned_tree, out_file=None, 
                           feature_names=X_train.columns, 
                           class_names=['No', 'Yes'], 
                           filled=True, rounded=True, 
                           special_characters=True)

# Visualizing the tree using Graphviz
graph = graphviz.Source(dot_data)
graph.render(filename='pruned_tree', format='svg')


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,0
1,1,1,1,38.0,1,0,71.2833,1
2,1,3,1,26.0,0,0,7.9250,0
3,1,1,1,35.0,1,0,53.1000,0
4,0,3,0,35.0,0,0,8.0500,0
5,0,3,0,28.0,0,0,8.4583,2
6,0,1,0,54.0,0,0,51.8625,0
7,0,3,0,2.0,3,1,21.0750,0
8,1,3,1,27.0,0,2,11.1333,0
9,1,2,1,14.0,1,0,30.0708,1


Training features shape: (712, 7)
Training target shape: (712,)
Test features shape: (179, 7)
Test target shape: (179,)


GridSearchCV(cv=10, estimator=DecisionTreeClassifier(random_state=10),
             n_jobs=-1,
             param_grid={'max_depth': [2, 3, 4, 5, 6, 7, 8],
                         'min_samples_split': [2, 3, 4, 5, 6, 7, 8]})

Best hyperparameters: {'max_depth': 5, 'min_samples_split': 2}
Best score: 0.8174491392801253


...

'tree.svg'

DecisionTreeClassifier(random_state=10)

DecisionTreeClassifier(ccp_alpha=0.003269350811485643, random_state=10)

Accuracy of pruned tree on the test data: 0.7430167597765364


'pruned_tree.svg'